In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Define custom objects dictionary
custom_objects = {"mse": tf.keras.losses.MeanSquaredError()}

# Load the saved model
model_gamma = tf.keras.models.load_model("best_NN_gamma.h5", custom_objects=custom_objects)
model_log = tf.keras.models.load_model("best_NN_lognormal.h5", custom_objects=custom_objects)

# Define the feature names (match the original dataset used to fit the scaler)
feature_names = ["r", "lambda", "D", "N", "T"]  # Update with actual names

# Load the fitted scaler
scaler_gamma = joblib.load("scaler_gamma.pkl")
scaler_log = joblib.load("scaler_log.pkl")

In [3]:
r0_grid = np.linspace(0,0.12,100001)
lambda_grid = np.linspace(25,45,100001)
D_grid = np.linspace(5,15,100001)*1e9

## Gamma Distribution

In [5]:
X_lam = pd.DataFrame({
    "r": np.full(len(lambda_grid),0.03),
    "lambda": lambda_grid,
    "D": np.full(len(lambda_grid),9e9),
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_lam = scaler_gamma.transform(X_lam)

price_lam = model_gamma.predict(X_scaled_lam, verbose=0).flatten()

delta_lam = np.diff(price_lam)

In [6]:
X_D = pd.DataFrame({
    "r": np.full(len(D_grid),0.03),
    "lambda": np.full(len(D_grid),35),
    "D": D_grid,
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_D = scaler_gamma.transform(X_D)

price_D = model_gamma.predict(X_scaled_D, verbose=0).flatten()

delta_D = np.diff(price_D)

In [7]:
X_r = pd.DataFrame({
    "r": r0_grid,
    "lambda": np.full(len(r0_grid),35),
    "D": np.full(len(r0_grid),9e9),
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_r = scaler_gamma.transform(X_r)

price_r = model_gamma.predict(X_scaled_r, verbose=0).flatten()

delta_r = np.diff(price_r)

In [8]:
eps_tol = 1e-6

rows = []

# lambda
violations = delta_lam > eps_tol

rows.append({
    "Severity": "Gamma",
    "Variable": "λ",
    "Expected sign": "Non-increasing",
    "Grid comparisons": len(delta_lam),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": np.max(delta[violations]) if np.any(violations) else 0.0
})

# D
violations = delta_D < -eps_tol

rows.append({
    "Severity": "Gamma",
    "Variable": "D",
    "Expected sign": "Non-decreasing",
    "Grid comparisons": len(delta_D),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": abs(np.min(delta[violations])) if np.any(violations) else 0.0
})

# r0
violations = delta_r > eps_tol

rows.append({
    "Severity": "Gamma",
    "Variable": "r0",
    "Expected sign": "Non-increasing",
    "Grid comparisons": len(delta_r),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": np.max(delta[violations]) if np.any(violations) else 0.0
})


results = pd.DataFrame(rows)

print(results)

  Severity Variable   Expected sign  Grid comparisons  Violation rate (%)  \
0    Gamma        λ  Non-increasing            100000                 0.0   
1    Gamma        D  Non-decreasing            100000                 0.0   
2    Gamma       r0  Non-increasing            100000                 0.0   

   Max violation  
0            0.0  
1            0.0  
2            0.0  


## Lognormal Distribution

In [10]:
X_lam = pd.DataFrame({
    "r": np.full(len(lambda_grid),0.03),
    "lambda": lambda_grid,
    "D": np.full(len(lambda_grid),9e9),
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_lam = scaler_log.transform(X_lam)

price_lam = model_log.predict(X_scaled_lam, verbose=0).flatten()

delta_lam = np.diff(price_lam)

In [11]:
X_D = pd.DataFrame({
    "r": np.full(len(D_grid),0.03),
    "lambda": np.full(len(D_grid),35),
    "D": D_grid,
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_D = scaler_log.transform(X_D)

price_D = model_log.predict(X_scaled_D, verbose=0).flatten()

delta_D = np.diff(price_D)

In [12]:
X_r = pd.DataFrame({
    "r": r0_grid,
    "lambda": np.full(len(r0_grid),35),
    "D": np.full(len(r0_grid),9e9),
    "N": np.full(len(lambda_grid),2),
    "T": np.full(len(lambda_grid),1)
})

X_scaled_r = scaler_log.transform(X_r)

price_r = model_log.predict(X_scaled_r, verbose=0).flatten()

delta_r = np.diff(price_r)

In [13]:
eps_tol = 1e-6

rows = []

# lambda
violations = delta_lam > eps_tol

rows.append({
    "Severity": "Lognormal",
    "Variable": "λ",
    "Expected sign": "Non-increasing",
    "Grid comparisons": len(delta_lam),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": np.max(delta[violations]) if np.any(violations) else 0.0
})

# D
violations = delta_D < -eps_tol

rows.append({
    "Severity": "Lognormal",
    "Variable": "D",
    "Expected sign": "Non-decreasing",
    "Grid comparisons": len(delta_D),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": abs(np.min(delta[violations])) if np.any(violations) else 0.0
})

# r0
violations = delta_r > eps_tol

rows.append({
    "Severity": "Lognormal",
    "Variable": "r0",
    "Expected sign": "Non-increasing",
    "Grid comparisons": len(delta_r),
    "Violation rate (%)": 100 * np.mean(violations),
    "Max violation": np.max(delta[violations]) if np.any(violations) else 0.0
})


results = pd.DataFrame(rows)

print(results)

    Severity Variable   Expected sign  Grid comparisons  Violation rate (%)  \
0  Lognormal        λ  Non-increasing            100000                 0.0   
1  Lognormal        D  Non-decreasing            100000                 0.0   
2  Lognormal       r0  Non-increasing            100000                 0.0   

   Max violation  
0            0.0  
1            0.0  
2            0.0  
